In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import gc
import zipfile

# =========================
# CONFIG
# =========================

DATA_PATH = "data/raw/CIC-ToN-IoT.csv"   # update this if the dataset file name changes

OUTPUT_DIR = Path("feature_check_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Set this if the label column name is already known.
# Leave it as None to detect the label column automatically.
LABEL_COL = None

# Keep this conservative for large datasets.
SAMPLE_SIZE = 100_000
CHUNK_SIZE = 100_000

MAX_MISSING_PERCENT = 30
MAX_INF_PERCENT = 30
QUASI_CONSTANT_THRESHOLD = 0.99

RANDOM_STATE = 42

In [2]:
def read_csv_sample(path, nrows=100_000):
    encodings = ["utf-8", "latin1", "ISO-8859-1"]
    last_error = None
    
    for enc in encodings:
        try:
            sample = pd.read_csv(
                path,
                nrows=nrows,
                encoding=enc,
                low_memory=False
            )
            return sample, enc
        except Exception as e:
            last_error = e
    
    raise RuntimeError(f"Failed to read CSV sample. Last error: {last_error}")


sample_df, USED_ENCODING = read_csv_sample(DATA_PATH, SAMPLE_SIZE)

print("Sample loaded successfully.")
print("Encoding used:", USED_ENCODING)
print("Sample shape:", sample_df.shape)

display(sample_df.head())

Sample loaded successfully.
Encoding used: utf-8
Sample shape: (100000, 85)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,177.30.87.144-192.168.1.1-0-0-0,177.30.87.144,0,192.168.1.1,0,0,25/04/2019 05:18:52 pm,47814343,5,0,...,1038036.0,0.000000e+00,1038036.0,1038036.0,5.187256e+14,8.984590e+14,1.556177e+15,1.657324e+07,0,Benign
1,167.49.176.28-50.165.192.168-0-0-0,167.49.176.28,0,50.165.192.168,0,0,25/04/2019 05:18:49 pm,2033142,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
2,230.158.52.59-177.21.192.168-0-0-0,230.158.52.59,0,177.21.192.168,0,0,25/04/2019 05:18:37 pm,82877133,14,0,...,1931160.5,1.711593e+06,3942470.0,226402.0,1.729085e+14,5.187256e+14,1.556177e+15,6.036493e+06,0,Benign
3,183.68.192.168-1.1.192.168-0-0-0,183.68.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,24359,2,0,...,0.0,0.000000e+00,0.0,0.0,1.556177e+15,0.000000e+00,1.556177e+15,1.556177e+15,0,Benign
4,183.41.192.168-1.1.192.168-0-0-0,183.41.192.168,0,1.1.192.168,0,0,25/04/2019 05:18:42 pm,10239351,3,0,...,4053975.0,0.000000e+00,4053975.0,4053975.0,7.780884e+14,1.100383e+15,1.556177e+15,6.185376e+06,0,Benign


In [3]:
print("Total columns:", len(sample_df.columns))
print("\nColumns:")
for i, col in enumerate(sample_df.columns, start=1):
    print(f"{i}. {col}")

possible_label_keywords = ["label", "class", "attack", "category", "type", "target"]

possible_label_cols = [
    col for col in sample_df.columns
    if any(keyword in col.lower() for keyword in possible_label_keywords)
]

print("\nPossible label columns:")
print(possible_label_cols)

for col in possible_label_cols:
    print("\n" + "=" * 60)
    print("Column:", col)
    print("Unique values in sample:", sample_df[col].nunique(dropna=False))
    print(sample_df[col].value_counts(dropna=False).head(30))

Total columns: 85

Columns:
1. Flow ID
2. Src IP
3. Src Port
4. Dst IP
5. Dst Port
6. Protocol
7. Timestamp
8. Flow Duration
9. Tot Fwd Pkts
10. Tot Bwd Pkts
11. TotLen Fwd Pkts
12. TotLen Bwd Pkts
13. Fwd Pkt Len Max
14. Fwd Pkt Len Min
15. Fwd Pkt Len Mean
16. Fwd Pkt Len Std
17. Bwd Pkt Len Max
18. Bwd Pkt Len Min
19. Bwd Pkt Len Mean
20. Bwd Pkt Len Std
21. Flow Byts/s
22. Flow Pkts/s
23. Flow IAT Mean
24. Flow IAT Std
25. Flow IAT Max
26. Flow IAT Min
27. Fwd IAT Tot
28. Fwd IAT Mean
29. Fwd IAT Std
30. Fwd IAT Max
31. Fwd IAT Min
32. Bwd IAT Tot
33. Bwd IAT Mean
34. Bwd IAT Std
35. Bwd IAT Max
36. Bwd IAT Min
37. Fwd PSH Flags
38. Bwd PSH Flags
39. Fwd URG Flags
40. Bwd URG Flags
41. Fwd Header Len
42. Bwd Header Len
43. Fwd Pkts/s
44. Bwd Pkts/s
45. Pkt Len Min
46. Pkt Len Max
47. Pkt Len Mean
48. Pkt Len Std
49. Pkt Len Var
50. FIN Flag Cnt
51. SYN Flag Cnt
52. RST Flag Cnt
53. PSH Flag Cnt
54. ACK Flag Cnt
55. URG Flag Cnt
56. CWE Flag Count
57. ECE Flag Cnt
58. Down/Up Ratio


In [4]:
def auto_select_label_column(df, possible_cols):
    if not possible_cols:
        return df.columns[-1]
    
    candidates = []
    
    for col in possible_cols:
        unique_count = df[col].nunique(dropna=False)
        name = col.lower()
        
        score = 0
        
        if "attack" in name:
            score += 5
        if "label" in name:
            score += 4
        if "class" in name:
            score += 3
        if "category" in name or "type" in name:
            score += 2
        
        # For multiclass data, prefer columns with 3 to 50 classes.
        if 3 <= unique_count <= 50:
            score += 10
        elif unique_count == 2:
            score += 3
        
        candidates.append((score, col, unique_count))
    
    candidates = sorted(candidates, reverse=True)
    return candidates[0][1]


if LABEL_COL is None:
    LABEL_COL = auto_select_label_column(sample_df, possible_label_cols)

print("Selected label column:", LABEL_COL)

if LABEL_COL not in sample_df.columns:
    raise ValueError(f"Label column '{LABEL_COL}' not found. Please check the column name.")

print("\nSample class distribution:")
display(sample_df[LABEL_COL].value_counts(dropna=False))

Selected label column: Attack

Sample class distribution:


Attack
Benign       99126
mitm           489
ddos           202
dos            145
scanning        29
injection        7
password         2
Name: count, dtype: int64

In [5]:
X_sample = sample_df.drop(columns=[LABEL_COL])
y_sample = sample_df[LABEL_COL]

def numeric_convert_ratio(series):
    non_null = series.dropna()
    
    if len(non_null) == 0:
        return 0.0
    
    if pd.api.types.is_numeric_dtype(non_null):
        return 1.0
    
    converted = pd.to_numeric(non_null, errors="coerce")
    return converted.notna().mean()


feature_basic = []

for col in X_sample.columns:
    s = X_sample[col]
    
    numeric_ratio = numeric_convert_ratio(s)
    is_numeric_candidate = numeric_ratio >= 0.95
    
    unique_count = s.nunique(dropna=False)
    unique_ratio = unique_count / max(len(s), 1)
    
    value_counts = s.value_counts(dropna=False, normalize=True)
    top_value_frequency = value_counts.iloc[0] if len(value_counts) > 0 else 1.0
    
    feature_basic.append({
        "feature": col,
        "sample_dtype": str(s.dtype),
        "sample_unique_count": unique_count,
        "sample_unique_ratio": unique_ratio,
        "sample_top_value_frequency": top_value_frequency,
        "numeric_convert_ratio": numeric_ratio,
        "is_numeric_candidate": is_numeric_candidate
    })

feature_basic_df = pd.DataFrame(feature_basic)

display(feature_basic_df.head())

,feature,sample_dtype,sample_unique_count,sample_unique_ratio,sample_top_value_frequency,numeric_convert_ratio,is_numeric_candidate
0,Flow ID,object,20372,0.20372,0.70859,0.0,False
1,Src IP,object,13791,0.13791,0.76647,0.0,False
2,Src Port,int64,5407,0.05407,0.70859,1.0,True
3,Dst IP,object,581,0.00581,0.76405,0.0,False
4,Dst Port,int64,658,0.00658,0.73083,1.0,True


In [6]:
def is_suspected_identifier(col):
    name = col.lower().strip()
    
    exact_bad = [
        "id", "flow id", "flow_id", "uid",
        "src ip", "src_ip", "source ip", "source_ip",
        "dst ip", "dst_ip", "destination ip", "destination_ip",
        "timestamp", "time stamp", "date"
    ]
    
    if name in exact_bad:
        return True
    
    bad_patterns = [
        "src ip", "dst ip", "source ip", "destination ip",
        "src_ip", "dst_ip", "source_ip", "destination_ip",
        "timestamp"
    ]
    
    if any(pattern in name for pattern in bad_patterns):
        return True
    
    # Do not automatically drop features containing the word "flow",
    # because features such as Flow Duration can still be valid.
    return False


suspected_identifier_cols = [
    col for col in X_sample.columns
    if is_suspected_identifier(col)
]

print("Suspected identifier or leakage-prone columns:")
for col in suspected_identifier_cols:
    print("-", col)

Suspected identifier or leakage-prone columns:
- Flow ID
- Src IP
- Dst IP
- Timestamp


In [7]:
all_columns = sample_df.columns.tolist()
feature_columns = [col for col in all_columns if col != LABEL_COL]

numeric_candidate_cols = feature_basic_df.loc[
    feature_basic_df["is_numeric_candidate"],
    "feature"
].tolist()

missing_counts = Counter()
inf_counts = Counter()
total_rows = 0
class_counts = Counter()

print("Start chunk profiling...")

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    encoding=USED_ENCODING,
    low_memory=False
)

for chunk_idx, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)
    
    # Class distribution
    if LABEL_COL in chunk.columns:
        class_counts.update(chunk[LABEL_COL].value_counts(dropna=False).to_dict())
    
    # Missing values
    missing_counts.update(chunk[feature_columns].isna().sum().to_dict())
    
    # Check infinite values only for candidate numeric features.
    for col in numeric_candidate_cols:
        if col in chunk.columns:
            numeric_col = pd.to_numeric(chunk[col], errors="coerce")
            inf_counts[col] += np.isinf(numeric_col).sum()
    
    if chunk_idx % 10 == 0:
        print(f"Processed chunks: {chunk_idx}, rows: {total_rows:,}")
    
    del chunk
    gc.collect()

print("Chunk profiling finished.")
print("Total rows:", total_rows)

Start chunk profiling...
Processed chunks: 10, rows: 1,000,000
Processed chunks: 20, rows: 2,000,000
Processed chunks: 30, rows: 3,000,000
Processed chunks: 40, rows: 4,000,000
Processed chunks: 50, rows: 5,000,000
Chunk profiling finished.
Total rows: 5351760


In [8]:
feature_summary = feature_basic_df.copy()

feature_summary["missing_count"] = feature_summary["feature"].map(lambda x: missing_counts.get(x, 0))
feature_summary["missing_percent"] = feature_summary["missing_count"] / total_rows * 100

feature_summary["inf_count"] = feature_summary["feature"].map(lambda x: inf_counts.get(x, 0))
feature_summary["inf_percent"] = feature_summary["inf_count"] / total_rows * 100

feature_summary["suspected_identifier"] = feature_summary["feature"].isin(suspected_identifier_cols)

feature_summary["is_quasi_constant"] = (
    feature_summary["sample_top_value_frequency"] >= QUASI_CONSTANT_THRESHOLD
)

feature_summary["recommended_drop_reason"] = ""

def add_reason(row):
    reasons = []
    
    if not row["is_numeric_candidate"]:
        reasons.append("non numeric or not safely convertible to numeric")
    
    if row["suspected_identifier"]:
        reasons.append("identifier or leakage prone column")
    
    if row["missing_percent"] > MAX_MISSING_PERCENT:
        reasons.append(f"missing values > {MAX_MISSING_PERCENT}%")
    
    if row["inf_percent"] > MAX_INF_PERCENT:
        reasons.append(f"infinite values > {MAX_INF_PERCENT}%")
    
    if row["is_quasi_constant"]:
        reasons.append("quasi constant feature")
    
    return "; ".join(reasons)

feature_summary["recommended_drop_reason"] = feature_summary.apply(add_reason, axis=1)

feature_summary["recommended_for_autoencoder"] = feature_summary["recommended_drop_reason"] == ""

feature_summary = feature_summary.sort_values(
    by=["recommended_for_autoencoder", "recommended_drop_reason", "feature"],
    ascending=[False, True, True]
)

display(feature_summary)

feature_summary.to_csv(OUTPUT_DIR / "feature_summary.csv", index=False)
print("Saved:", OUTPUT_DIR / "feature_summary.csv")

,feature,sample_dtype,sample_unique_count,sample_unique_ratio,sample_top_value_frequency,numeric_convert_ratio,is_numeric_candidate,missing_count,missing_percent,inf_count,inf_percent,suspected_identifier,is_quasi_constant,recommended_drop_reason,recommended_for_autoencoder
53,ACK Flag Cnt,int64,255,0.00255,0.45756,1.0,True,0,0.0,0,0.0,False,False,,True
77,Active Max,float64,3266,0.03266,0.92850,1.0,True,0,0.0,0,0.0,False,False,,True
75,Active Mean,float64,3327,0.03327,0.92850,1.0,True,0,0.0,0,0.0,False,False,,True
78,Active Min,float64,2237,0.02237,0.92850,1.0,True,0,0.0,0,0.0,False,False,,True
76,Active Std,float64,1622,0.01622,0.97772,1.0,True,0,0.0,0,0.0,False,False,,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,Fwd URG Flags,int64,1,0.00001,1.00000,1.0,True,0,0.0,0,0.0,False,True,quasi constant feature,False
83,Label,int64,2,0.00002,0.99126,1.0,True,0,0.0,0,0.0,False,True,quasi constant feature,False
51,RST Flag Cnt,int64,17,0.00017,0.99598,1.0,True,0,0.0,0,0.0,False,True,quasi constant feature,False
69,Subflow Bwd Pkts,int64,1,0.00001,1.00000,1.0,True,0,0.0,0,0.0,False,True,quasi constant feature,False


Saved: feature_check_outputs\feature_summary.csv


In [9]:
candidate_features = feature_summary.loc[
    feature_summary["recommended_for_autoencoder"],
    "feature"
].tolist()

dropped_features = feature_summary.loc[
    ~feature_summary["recommended_for_autoencoder"],
    ["feature", "recommended_drop_reason"]
]

print("Total recommended features for Autoencoder:", len(candidate_features))
print("\nRecommended features:")
for i, col in enumerate(candidate_features, start=1):
    print(f"{i}. {col}")

print("\nDropped or excluded features:")
display(dropped_features)

Total recommended features for Autoencoder: 67

Recommended features:
1. ACK Flag Cnt
2. Active Max
3. Active Mean
4. Active Min
5. Active Std
6. Bwd Blk Rate Avg
7. Bwd Byts/b Avg
8. Bwd Header Len
9. Bwd IAT Max
10. Bwd IAT Mean
11. Bwd IAT Min
12. Bwd IAT Std
13. Bwd IAT Tot
14. Bwd Pkt Len Max
15. Bwd Pkt Len Mean
16. Bwd Pkt Len Min
17. Bwd Pkt Len Std
18. Bwd Pkts/b Avg
19. Bwd Pkts/s
20. Bwd Seg Size Avg
21. Down/Up Ratio
22. Dst Port
23. FIN Flag Cnt
24. Flow Byts/s
25. Flow Duration
26. Flow IAT Max
27. Flow IAT Mean
28. Flow IAT Min
29. Flow IAT Std
30. Flow Pkts/s
31. Fwd Act Data Pkts
32. Fwd Header Len
33. Fwd IAT Max
34. Fwd IAT Mean
35. Fwd IAT Min
36. Fwd IAT Std
37. Fwd IAT Tot
38. Fwd Pkt Len Max
39. Fwd Pkt Len Mean
40. Fwd Pkt Len Min
41. Fwd Pkt Len Std
42. Fwd Pkts/s
43. Fwd Seg Size Avg
44. Fwd Seg Size Min
45. Idle Max
46. Idle Mean
47. Idle Min
48. Idle Std
49. Init Bwd Win Byts
50. Init Fwd Win Byts
51. PSH Flag Cnt
52. Pkt Len Max
53. Pkt Len Mean
54. Pkt Len

,feature,recommended_drop_reason
3,Dst IP,non numeric or not safely convertible to numer...
0,Flow ID,non numeric or not safely convertible to numer...
1,Src IP,non numeric or not safely convertible to numer...
6,Timestamp,non numeric or not safely convertible to numer...
37,Bwd PSH Flags,quasi constant feature
39,Bwd URG Flags,quasi constant feature
55,CWE Flag Count,quasi constant feature
56,ECE Flag Cnt,quasi constant feature
63,Fwd Blk Rate Avg,quasi constant feature
61,Fwd Byts/b Avg,quasi constant feature


In [10]:
# Save candidate features
with open(OUTPUT_DIR / "candidate_features.txt", "w", encoding="utf-8") as f:
    for col in candidate_features:
        f.write(col + "\n")

# Save dropped features
dropped_features.to_csv(OUTPUT_DIR / "dropped_features.csv", index=False)

# Save class distribution
class_distribution = pd.DataFrame({
    "class": list(class_counts.keys()),
    "count": list(class_counts.values())
})

class_distribution["percentage"] = class_distribution["count"] / class_distribution["count"].sum() * 100
class_distribution = class_distribution.sort_values(by="count", ascending=False)

display(class_distribution)

class_distribution.to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)

print("Saved candidate_features.txt")
print("Saved dropped_features.csv")
print("Saved class_distribution.csv")

,class,count,percentage
0,Benign,2515236,46.998296
9,xss,2149308,40.160770
6,password,340208,6.356937
5,injection,277696,5.188872
4,scanning,36205,0.676506
7,backdoor,27145,0.507216
8,ransomware,5098,0.095258
1,mitm,517,0.009660
2,ddos,202,0.003774
3,dos,145,0.002709


Saved candidate_features.txt
Saved dropped_features.csv
Saved class_distribution.csv


In [11]:
min_class_count = class_distribution["count"].min()
num_classes = class_distribution.shape[0]

print("Number of classes:", num_classes)
print("Minimum class count:", min_class_count)

if min_class_count < 2:
    print("Warning: At least one class has fewer than 2 samples. Stratified split may fail.")
else:
    print("Stratified split should be possible.")

Number of classes: 10
Minimum class count: 145
Stratified split should be possible.


In [12]:
report_path = OUTPUT_DIR / "feature_check_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("FEATURE CHECK REPORT\n")
    f.write("====================\n\n")
    
    f.write(f"Dataset path: {DATA_PATH}\n")
    f.write(f"Total rows: {total_rows}\n")
    f.write(f"Total columns: {len(all_columns)}\n")
    f.write(f"Selected label column: {LABEL_COL}\n")
    f.write(f"Number of classes: {num_classes}\n\n")
    
    f.write("Class distribution:\n")
    for _, row in class_distribution.iterrows():
        f.write(f"{row['class']}: {row['count']} ({row['percentage']:.4f}%)\n")
    
    f.write("\n")
    f.write(f"Numeric candidate columns: {len(numeric_candidate_cols)}\n")
    f.write(f"Suspected identifier columns: {len(suspected_identifier_cols)}\n")
    f.write(f"Recommended Autoencoder features: {len(candidate_features)}\n")
    f.write(f"Dropped or excluded features: {len(dropped_features)}\n\n")
    
    f.write("Recommended features for Autoencoder:\n")
    for col in candidate_features:
        f.write(f"- {col}\n")
    
    f.write("\nDropped or excluded features:\n")
    for _, row in dropped_features.iterrows():
        f.write(f"- {row['feature']}: {row['recommended_drop_reason']}\n")

print("Saved report:", report_path)

Saved report: feature_check_outputs\feature_check_report.txt


In [ ]:
zip_path = Path("feature_check_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in OUTPUT_DIR.glob("*"):
        zipf.write(file_path, arcname=file_path.name)

print("Created:", zip_path)

In [13]:
# =========================
# 14. SAVE TEXT REPORT
# =========================

report_path = OUTPUT_DIR / "feature_check_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("FEATURE CHECK REPORT\n")
    f.write("====================\n\n")
    
    f.write(f"Dataset shape: {df.shape}\n")
    f.write(f"Selected label column: {LABEL_COL}\n")
    f.write(f"Number of classes: {y.nunique()}\n")
    f.write(f"Duplicate rows: {duplicate_rows}\n\n")
    
    f.write("Class distribution:\n")
    f.write(str(y.value_counts(dropna=False)))
    f.write("\n\n")
    
    f.write(f"Numeric feature columns: {len(numeric_cols)}\n")
    f.write(f"Categorical or non-numeric feature columns: {len(categorical_cols)}\n")
    f.write(f"Suspected identifier columns: {len(suspected_identifier_cols)}\n")
    f.write(f"Constant features: {len(constant_features)}\n")
    f.write(f"High missing features > {MAX_MISSING_PERCENT}%: {len(high_missing_features)}\n")
    f.write(f"Infinite value features: {len(inf_features)}\n")
    f.write(f"Initial candidate features: {len(candidate_features)}\n\n")
    
    f.write("Categorical or non-numeric columns:\n")
    f.write("\n".join(categorical_cols))
    f.write("\n\n")
    
    f.write("Suspected identifier columns:\n")
    f.write("\n".join(suspected_identifier_cols))
    f.write("\n\n")
    
    f.write("Constant features:\n")
    f.write("\n".join(constant_features))
    f.write("\n\n")
    
    f.write("High missing features:\n")
    f.write("\n".join(high_missing_features))
    f.write("\n\n")
    
    f.write("Infinite value features:\n")
    f.write("\n".join(inf_features))
    f.write("\n\n")
    
    f.write("Candidate features:\n")
    f.write("\n".join(candidate_features))

print("Report saved to:", report_path)
print("All outputs saved in:", OUTPUT_DIR)

NameError: name 'df' is not defined

In [1]:
print("Shape:", df.shape)
print("Columns:", df.shape[1])
print("Rows:", df.shape[0])
print("Classes:", df['Attack'].nunique())
df.head()

NameError: name 'df' is not defined